In [ ]:
import subprocess
import warnings
warnings.filterwarnings('ignore')
import re
import os
import pandas as pd
import duckdb
import shutil
import json
import os
import snowflake.connector

In [ ]:
project_dir = r"C:\EnterpriseCloud\astro_datautility\dags\dbt\dbt_tel" 
old_path = r'models\tmea_product_support_group'
new_path = r'models\tmea_quality'

old_path = os.path.join(project_dir,old_path)
new_path = os.path.join(project_dir,new_path)

old_schema,new_schema = 'TMEA_PRODUCT_SUPPORT_GROUP','TMEA_QUALITY'

pub_de = ['SILVER__TMEA_PRODUCT_SUPPORT_GROUP__ACCESSLOGS',
]


In [ ]:
#2.LOAD DBT OBJECT
from dbt.cli.main import dbtRunner,dbtRunnerResult
from dbt.contracts.graph.manifest import Manifest
cli_args = ["deps","--project-dir",project_dir]
res = dbtRunner().invoke(cli_args)
res: dbtRunnerResult = dbtRunner().invoke(["parse","--project-dir",project_dir])
manifest: Manifest = res.result

dbt = dbtRunner(manifest=manifest)

In [ ]:
#3.RUN dbt list and generate the lineage of all the reports
final_df_models = pd.DataFrame()
final_df_src = pd.DataFrame()

for view in pub_de:
    cli_args = ["list","--select",f"+{view}","--project-dir",project_dir]
    res = dbt.invoke(cli_args)
    list_res = []
    for r in res.result:
        list_res.append(r)
    #sources list
    de_src = [src for src in list_res if "source:" in src]

    #models list
    de_models = []
    for model in list_res:
        if "source:" in model:
            break
        de_models.append(model) 

    #test
    de_test = []
    for test in reversed(list_res):
        if "source:" in test:
            break
        de_test.append(test)

    de_src = [i.split(".") for i in de_src]
    de_test = [i.split(".") for i in de_test]
    de_models = [i.split(".") for i in de_models]

    df_models = pd.DataFrame(de_models,columns=['dbt project','database','schema','model'])
    df_src = pd.DataFrame(de_src,columns=['dbt project','schema','table'])
    #df_test = pd.DataFrame(de_test,columns=['dbt project','database','schema','test'])
    
    df_models.drop('dbt project', axis=1, inplace=True)
    df_src.drop('dbt project', axis=1, inplace=True)

    final_df_models = pd.concat([final_df_models, df_models], ignore_index=True)
    final_df_src = pd.concat([final_df_src, df_src], ignore_index=True)
    

In [ ]:
final_df_models['new_name'] = final_df_models['model'].str.replace(old_schema,new_schema)

In [ ]:
new_name = {}
for id,row in final_df_models.iterrows():
    new_name[row['model']] = row['new_name']

In [ ]:
with open(os.path.join(project_dir,"target\\manifest.json"), 'r') as f:
    manifest = json.load(f)

In [ ]:
snowfiles = []
for key,value in manifest['nodes'].items():
    if value['name'] in final_df_models['model'].to_list():
        new_file_name = value['name'].replace(f'__{old_schema}__',f'__{new_schema}__')
        new_sql  = value['raw_code']
        snowfiles.append(new_file_name)
        if 'BRONZE__' in value['name']:
            shutil.copy(os.path.join(old_path,'bronze',f'{value["name"]}.SQL'),os.path.join(new_path,'bronze',f'{new_file_name}.SQL'))
            continue

        for ref in value['refs']:
            ref_model_name  = ref['name']
            new_ref_model_name = new_name.get(ref_model_name)
            new_ref = "{{ ref('"+new_ref_model_name+"') }}"
            ref_regex = '''\\{\\{ *ref *\\( *['"] *'''+ref_model_name+''' *['"] *\\) *\\}\\}'''
            new_sql = re.sub(ref_regex, new_ref, new_sql)
        
        db_folder = 'gold' if 'GOLD__' in value['name'] else 'silver'
        with open(os.path.join(new_path,db_folder,f'{new_file_name}.SQL'), 'w+', encoding='utf-8') as f:
            f.write("\n".join(new_sql.splitlines()))



In [ ]:
query = ''
for file in snowfiles:
    if (db:='bronze') and file.lower().startswith(db):
        with open(os.path.join(project_dir,new_path,db,f'{file}.SQL'),'r') as filex:
            data = filex.read()
    elif (db:='silver') and file.lower().startswith(db):
        with open(os.path.join(project_dir,new_path,db,f'\{file}.SQL'),'r') as filex:
            data = filex.read()
    elif (db:='gold') and file.lower().startswith(db):
        with open(os.path.join(project_dir,new_path,db,f'\{file}.SQL'),'r') as filex:
            data = filex.read()
            
    match = re.search(r"alias\s*=\s*'([^']+)'", data)
    if match:
        alias = match.group(1)
        query += f"\n SELECT GET_DDL('TABLE','{db}.TMEA_PRODUCT_SUPPORT_GROUP.{file.split('__')[-1]}') UNION ALL"



In [ ]:
print(query)